# Dense Retrieval Eval (Titan + FAISS)

Mirrors `bm25_eval.ipynb`, but retrieval is **semantic**: the query is
embedded with the same Titan model as the chunks, and FAISS returns the
nearest vectors by cosine similarity. Scored with the identical metrics so
the numbers line up directly against BM25.

**Prereqs:** `rag_eval/` uploaded next to this notebook, and
`data_processing.ipynb` run through its **Dense ingestion** section (so
`indexes/faiss.index` + `indexes/id_map.parquet` exist in S3).

## 1. Imports + session

In [ ]:
import sys
!{sys.executable} -m pip install -q faiss-cpu==1.15.1

sys.path.insert(0, ".")  # rag_eval/ next to this notebook
import io, os, tempfile, time
import boto3, numpy as np, pandas as pd, faiss

REGION = "us-east-1"
BUCKET = "rag-arch-eval-data-875699230934"
INDEXES_PREFIX = "indexes"
DATA_PREFIX = "data"

s3 = boto3.client("s3", region_name=REGION)
def read_parquet_s3(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))

## 2. Load the FAISS index + id_map from S3

Download the serialized index to a temp file and `faiss.read_index` it; load
the id_map that turns a FAISS row position back into a doc_id. These must be
the pair written together in ingestion (same order).

In [ ]:
with tempfile.TemporaryDirectory() as d:
    local = os.path.join(d, "faiss.index")
    s3.download_file(BUCKET, f"{INDEXES_PREFIX}/faiss.index", local)
    index = faiss.read_index(local)

id_map = read_parquet_s3(f"{INDEXES_PREFIX}/id_map.parquet").sort_values("row")
row_to_doc = id_map["doc_id"].tolist()  # position i -> doc_id
print("index size:", index.ntotal, "| id_map rows:", len(row_to_doc))
assert index.ntotal == len(row_to_doc), "index/id_map length mismatch"

## 3. Load queries (exclude null)

Same as BM25: drop `null_query` (no gold evidence), dedupe evidence titles.

In [ ]:
q_df = read_parquet_s3(f"{DATA_PREFIX}/queries.parquet")
qrecs = []
for r in q_df.itertuples():
    rel = tuple(dict.fromkeys(e["title"] for e in r.evidence_list))
    if rel:
        qrecs.append({"query": r.query, "question_type": r.question_type, "rel": set(rel)})
print("non-null queries:", len(qrecs))

## 4. Embed queries + search

Each query is embedded with Titan (same model as the chunks — this symmetry
is what makes the vectors comparable), normalized, then searched against
FAISS. We fetch extra chunks and collapse to unique doc_ids so the top-k
*documents* fill up. Query embeddings are cached by `TitanEmbedder`, but here
we call Bedrock directly for clarity on what's happening per query.

In [ ]:
from rag_eval.bedrock import TitanEmbedder
embedder = TitanEmbedder(model_id="amazon.titan-embed-text-v2:0", region=REGION)

def normalize(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n[n == 0] = 1.0
    return m / n

def dense_doc_ids(query, fetch):
    q = normalize(embedder.embed_query(query).reshape(1, -1).astype(np.float32))
    _, idx = index.search(q, min(fetch, index.ntotal))
    seen, out = set(), []
    for i in idx[0]:
        if i == -1:
            continue
        d = row_to_doc[i]
        if d not in seen:
            seen.add(d); out.append(d)
    return out

# quick smoke on one query
print(dense_doc_ids(qrecs[0]["query"], 10)[:5])

## 5. Score every query

Same metrics as BM25. This embeds every query (a few thousand Titan calls) —
a minute or two; cached on re-run.

In [ ]:
from rag_eval.metrics import METRICS
KS = [1, 3, 5, 10, 20]
FETCH = max(KS) * 5

rows = []
for j, q in enumerate(qrecs):
    ranked = dense_doc_ids(q["query"], FETCH)
    row = {"question_type": q["question_type"], "n_relevant": len(q["rel"])}
    for k in KS:
        for name, fn in METRICS.items():
            row[f"{name}@{k}"] = fn(ranked, q["rel"], k)
    rows.append(row)
    if (j + 1) % 500 == 0:
        print(f"  {j+1}/{len(qrecs)} queries scored")

per_q = pd.DataFrame(rows)
print("scored", len(per_q), "queries")

## 6. Results + compare to BM25

Print the same overall + per-type breakdown. Put these side by side with the
BM25 numbers from `bm25_eval.ipynb` — dense usually wins on the reworded /
inference queries (semantic match), BM25 can win where exact keywords matter.

In [ ]:
metric_cols = [c for c in per_q.columns if "@" in c]
print("=== DENSE OVERALL ===")
print(per_q[metric_cols].mean().round(4).to_string())

print("\n=== DENSE BY QUESTION TYPE ===")
by_type = per_q.groupby("question_type")[metric_cols].mean().round(4)
print(by_type[["recall@10", "all_hit@10", "mrr@10", "ndcg@10"]].to_string())

## 7. (optional) Save results to S3

In [ ]:
RUN_NAME = "dense_w256_o32_notebook"
summary = {"run": RUN_NAME, "n_queries": len(per_q), "n_chunks": index.ntotal}
summary.update(per_q[metric_cols].mean().round(4).to_dict())
for df, key in [
    (per_q, f"results/notebook/{RUN_NAME}_per_query.csv"),
    (pd.DataFrame([summary]), f"results/notebook/{RUN_NAME}_summary.csv"),
]:
    b = io.StringIO(); df.to_csv(b, index=False)
    s3.put_object(Bucket=BUCKET, Key=key, Body=b.getvalue().encode())
    print("wrote s3://%s/%s" % (BUCKET, key))